# Problem 2: Bundle Adjustment SLAM with GTSAM
All of your code goes in `P2_slam.py`. GTSAM has no pip package for Windows; if you are on Windows, run this notebook in Google Colab (uncomment the next cell) or on your Ubuntu VM.

In [ ]:
# Google Colab only: get the repo and install GTSAM. Edit P2_slam.py in the Colab file
# editor and download it before the runtime resets, or your changes are lost.
# !git clone https://github.com/PrinciplesofRobotAutonomy/AA274a-HW4-F26
# %cd AA274a-HW4-F26
# !pip install gtsam

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
import gtsam
import gtsam.utils.plot as gtsam_plot
from gtsam.symbol_shorthand import L, X

from P2_slam import add_prior, odometry_pose, add_odometry, add_measurements, \
    landmark_from_measurement, initial_estimate

os.makedirs("plots", exist_ok=True)
truth = np.load("data/ground_truth.npz")
true_poses, true_landmarks = truth["poses"], truth["landmarks"]
num_landmarks = len(true_landmarks)

# Noise models: odometry (x, y, theta), measurements (bearing, range)
PRIOR_NOISE = gtsam.noiseModel.Diagonal.Sigmas([0.01, 0.01, np.deg2rad(0.5)])
ODOMETRY_NOISE = gtsam.noiseModel.Diagonal.Sigmas([0.2, 0.05, np.deg2rad(5.0)])
MEASUREMENT_NOISE = gtsam.noiseModel.Diagonal.Sigmas([np.deg2rad(1.0), 0.05])

# Loop closure: at X(8) the robot recognizes the spot it passed at X(2), 6 m to its left, facing the other way
LOOP_I, LOOP_J, LOOP_POSE = 2, 8, gtsam.Pose2(0.0, 6.0, np.pi)


def load_history(max_range):
    """Returns speeds, turns (one per step) and (T, M) ranges, bearings in degrees (NaN if not seen)."""
    d = np.genfromtxt(f"data/robot_history_{max_range}.csv", delimiter=",", names=True)
    ranges = np.stack([d[f"range_{j}"] for j in range(num_landmarks)], axis=1)
    bearings = np.stack([d[f"bearing_{j}"] for j in range(num_landmarks)], axis=1)
    return d["speed"][:-1], d["turn"][:-1], ranges, bearings


def build_graph(max_range, loop_closure=True):
    speeds, turns, ranges, bearings = load_history(max_range)
    graph = gtsam.NonlinearFactorGraph()
    add_prior(graph, PRIOR_NOISE)
    add_odometry(graph, speeds, turns, ODOMETRY_NOISE)
    add_measurements(graph, ranges, bearings, MEASUREMENT_NOISE)
    if loop_closure:
        graph.add(gtsam.BetweenFactorPose2(X(LOOP_I), X(LOOP_J), LOOP_POSE, ODOMETRY_NOISE))
    return graph, initial_estimate(speeds, turns, ranges, bearings)


def pose_errors(values):
    """Position error (m) of every pose against the ground truth."""
    return np.array([np.hypot(values.atPose2(X(i)).x() - p[0], values.atPose2(X(i)).y() - p[1])
                     for i, p in enumerate(true_poses)])


def plot_solution(ax, graph, values, title, marginals=None):
    plt.sca(ax)
    for i in range(len(true_poses)):
        cov = marginals.marginalCovariance(X(i)) if marginals else None
        gtsam_plot.plot_pose2(ax.figure.number, values.atPose2(X(i)), 0.5, cov)
    for j in range(num_landmarks):
        if values.exists(L(j)):
            cov = marginals.marginalCovariance(L(j)) if marginals else None
            gtsam_plot.plot_point2(ax.figure.number, values.atPoint2(L(j)), "b", cov)
    xy = np.array([[values.atPose2(X(i)).x(), values.atPose2(X(i)).y()] for i in range(len(true_poses))])
    ax.plot(xy[:, 0], xy[:, 1], "k:", lw=1)
    ax.plot(true_poses[:, 0], true_poses[:, 1], "g--", lw=1, label="true path")
    ax.plot(true_landmarks[:, 0], true_landmarks[:, 1], "rx", label="true landmarks")
    ax.set_title(title)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.axis("equal")

## Sensor range 5 m

In [ ]:
graph, initial = build_graph(5)
print("number of factors:", graph.size())

result = gtsam.LevenbergMarquardtOptimizer(graph, initial).optimize()
marginals = gtsam.Marginals(graph, result)

e0, e1 = pose_errors(initial), pose_errors(result)
print(f"dead reckoning: max position error {e0.max():.3f} m, final {e0[-1]:.3f} m")
print(f"optimized:      max position error {e1.max():.3f} m, final {e1[-1]:.3f} m")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_solution(axes[0], graph, initial, "initial estimate (dead reckoning)")
plot_solution(axes[1], graph, result, "optimized, with 95% confidence ellipses", marginals)
axes[1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig("plots/p2_slam_5.png", dpi=120)
plt.show()

## Effect of the sensor range and the loop closure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, max_range in zip(axes, [3, 5, 10]):
    graph, initial = build_graph(max_range)
    result = gtsam.LevenbergMarquardtOptimizer(graph, initial).optimize()
    marginals = gtsam.Marginals(graph, result)
    sigma = np.sqrt(np.trace(marginals.marginalCovariance(X(len(true_poses) - 1))[:2, :2]))
    print(f"range {max_range:2d} m: {graph.size():3d} factors, max position error {pose_errors(result).max():.3f} m, "
          f"final pose sigma {sigma:.3f} m")
    plot_solution(ax, graph, result, f"sensor range {max_range} m", marginals)
plt.tight_layout()
plt.savefig("plots/p2_ranges.png", dpi=120)
plt.show()

print()
for max_range in [3, 5, 10]:
    for loop_closure in [True, False]:
        graph, initial = build_graph(max_range, loop_closure)
        result = gtsam.LevenbergMarquardtOptimizer(graph, initial).optimize()
        sigma = np.sqrt(np.trace(gtsam.Marginals(graph, result).marginalCovariance(X(len(true_poses) - 1))[:2, :2]))
        print(f"range {max_range:2d} m, loop closure {str(loop_closure):5s}: final pose sigma {sigma:.3f} m")